[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.5_cold_start/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.5_cold_start/lab.ipynb)

# 7.5 Lab: Cold Start in Serverless LLM Serving[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.5_cold_start/lab.ipynb)[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.5_cold_start/lab.ipynb)This lab models cold start latency breakdown and compares mitigation strategies for different model sizes and infrastructure choices.

In [ ]:
# Install required packages via subprocessimport subprocessimport sys# numpy for numerical computationsubprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib"])

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# --- Cold Start Component Parameters ---# Model sizes in GB (weight files on disk)MODEL_SIZES_GB = [3.5, 7.5, 16, 35, 70, 140]# Labels for each model sizeMODEL_LABELS = ["Llama 1B\n(4-bit)", "Llama 8B\n(4-bit)", "Llama 8B\n(FP16)", "Llama 70B\n(4-bit)", "Llama 70B\n(FP16)", "Llama 405B\n(4-bit)"]# Network bandwidth from object storage in GB/sNETWORK_BANDWIDTH_GBPS = 10.0# NVMe read speed in GB/s (local SSD cache)NVME_BANDWIDTH_GBPS = 3.5# PCIe bandwidth for CPU-to-GPU transfer in GB/sPCIE_BANDWIDTH_GBPS = 32.0# Fixed container pull time in secondsCONTAINER_PULL_S = 8.0# Fixed runtime initialization time in secondsRUNTIME_INIT_S = 3.0# Fixed CUDA warmup time in secondsWARMUP_S = 2.0

## Experiment 1: Cold Start Breakdown by Model SizeBreaking down where cold start time goes for each model size. The dominant cost shifts from fixed overhead (container, runtime) for small models to weight transfer for large models.

In [ ]:
def compute_cold_start_breakdown(model_size_gb, net_bw, nvme_bw, pcie_bw, container_s, runtime_s, warmup_s):    """Compute cold start time breakdown for a single model.        Returns dict with time for each phase.    """    # Time to download weights from S3/object storage    download_time = model_size_gb / (net_bw / 8)  # Convert Gbps to GB/s    # Time to load from local NVMe cache (alternative path)    nvme_load_time = model_size_gb / nvme_bw    # Time to transfer from CPU RAM to GPU over PCIe    gpu_transfer_time = model_size_gb / pcie_bw    # Return breakdown as dictionary    return {        "container": container_s,       # Fixed: pull container image        "runtime": runtime_s,           # Fixed: init Python + CUDA        "download": download_time,      # Variable: fetch weights from network        "gpu_transfer": gpu_transfer_time,  # Variable: CPU RAM -> GPU        "warmup": warmup_s,            # Fixed: CUDA graph compilation        "nvme_path": nvme_load_time,   # Alternative: load from local cache    }# Compute breakdown for each model size# Store results for plottingbreakdowns = []for size in MODEL_SIZES_GB:    # Compute the cold start breakdown for this model    bd = compute_cold_start_breakdown(        size, NETWORK_BANDWIDTH_GBPS, NVME_BANDWIDTH_GBPS,        PCIE_BANDWIDTH_GBPS, CONTAINER_PULL_S, RUNTIME_INIT_S, WARMUP_S    )    # Append to results list    breakdowns.append(bd)# Extract components for stacked bar chart# Container pull time for each model (constant)containers = [b["container"] for b in breakdowns]# Runtime init time for each model (constant)runtimes = [b["runtime"] for b in breakdowns]# Network download time for each model (scales with size)downloads = [b["download"] for b in breakdowns]# GPU transfer time for each model (scales with size)gpu_transfers = [b["gpu_transfer"] for b in breakdowns]# Warmup time for each model (constant)warmups = [b["warmup"] for b in breakdowns]

In [ ]:
# Plot stacked bar chart showing cold start breakdownfig_3, ax_3 = plt.subplots(figsize=(12, 6))# X positions for barsx = np.arange(len(MODEL_LABELS))# Bar widthwidth = 0.6# Stack each component on top of the previous one# Bottom accumulator for stacking barsbottom = np.zeros(len(MODEL_LABELS))# Container pull: grayax_3.bar(x, containers, width, bottom=bottom, label='Container Pull', color='#f3f4f6', edgecolor='#000')# Update bottom for next stack levelbottom += containers# Runtime init: blueax_3.bar(x, runtimes, width, bottom=bottom, label='Runtime Init', color='#dbeafe', edgecolor='#000')# Update bottom for next stack levelbottom += runtimes# Weight download: amber (the dominant cost for large models)ax_3.bar(x, downloads, width, bottom=bottom, label='Weight Download (S3)', color='#fef3c7', edgecolor='#000')# Update bottom for next stack levelbottom += downloads# GPU transfer: orangeax_3.bar(x, gpu_transfers, width, bottom=bottom, label='GPU Transfer (PCIe)', color='#ffedd5', edgecolor='#000')# Update bottom for next stack levelbottom += gpu_transfers# Warmup: greenax_3.bar(x, warmups, width, bottom=bottom, label='CUDA Warmup', color='#dcfce7', edgecolor='#000')# Add total time labels on top of each bar# Calculate total cold start time for annotationtotals = [c + r + d + g + w for c, r, d, g, w in zip(containers, runtimes, downloads, gpu_transfers, warmups)]for i, total in enumerate(totals):    # Annotate each bar with total seconds    ax_3.text(i, total + 0.5, f'{total:.0f}s', ha='center', fontweight='bold')# Formatting the axes_3 and titleax_3.set_xticks(x)ax_3.set_xticklabels(MODEL_LABELS)ax_3.set_ylabel('Cold Start Time (seconds)')ax_3.set_title('Cold Start Breakdown by Model Size (Network Download Path)')ax_3.legend(loc='upper left')ax_3.grid(True, axis='y', alpha=0.3)plt.tight_layout()# Save figure to disk for referenceplt.savefig('cold_start_breakdown.png', dpi=150, bbox_inches='tight')plt.show()# Print the total cold start for the most common production modelprint(f"Llama 8B FP16 cold start: {totals[2]:.1f}s")print(f"Llama 70B 4-bit cold start: {totals[3]:.1f}s")

## Experiment 2: Mitigation Strategy ComparisonCompare total cold start under different strategies: no optimization, NVMe caching, snapshot loading, and pre-warming.

In [ ]:
# --- Mitigation strategy parameters ---# Target model for comparison: Llama 8B FP16 (16 GB)TARGET_MODEL_GB = 16.0# Snapshot restore time (pre-serialized GPU state)SNAPSHOT_RESTORE_S = 5.0# Pre-warm time (always 0 because replicas are already running)PREWARM_TIME_S = 0.0# Monthly GPU cost for keeping one replica warm ($)MONTHLY_GPU_COST = 2500.0# Hours per month the replica would be idleIDLE_HOURS_PER_MONTH = 500.0def compute_strategy_costs(model_gb, strategies_config):    """Compute cold start latency and monthly cost for each strategy."""    # Store results as list of dicts    results = []    for name, config in strategies_config.items():        # Calculate total cold start latency for this strategy        latency = config["latency_fn"](model_gb)        # Calculate monthly cost overhead for this strategy        cost = config["monthly_cost"]        # Store result tuple        results.append({"name": name, "latency": latency, "cost": cost})    return results# Define each mitigation strategy with its latency function and coststrategies = {    "No Optimization": {        # Full download path: container + runtime + download + gpu + warmup        "latency_fn": lambda gb: 8 + 3 + gb / (10/8) + gb / 32 + 2,        # No additional infrastructure cost        "monthly_cost": 0,    },    "NVMe Cache": {        # Skip network download, read from local SSD instead        "latency_fn": lambda gb: 8 + 3 + gb / 3.5 + gb / 32 + 2,        # Cost of NVMe storage: ~$100/month for 2TB        "monthly_cost": 100,    },    "Snapshot Loading": {        # Restore pre-serialized state from NVMe (no parsing/compilation)        "latency_fn": lambda gb: SNAPSHOT_RESTORE_S + gb / 32,        # Cost of snapshot storage + NVMe        "monthly_cost": 150,    },    "Pre-warming (1 replica)": {        # Zero cold start: replica is already running        "latency_fn": lambda gb: 0,        # Pay for one always-on GPU replica        "monthly_cost": MONTHLY_GPU_COST * (IDLE_HOURS_PER_MONTH / 730),    },}# Compute results for each strategyresults = compute_strategy_costs(TARGET_MODEL_GB, strategies)# Extract values for plotting# Strategy names for x-axis labelsnames = [r["name"] for r in results]# Cold start latency in secondslatencies = [r["latency"] for r in results]# Monthly cost in dollarscosts = [r["cost"] for r in results]

In [ ]:
# Plot dual-axis chart: latency vs cost for each strategyfig, ax1 = plt.subplots(figsize=(10, 5))# Bar chart for latency on primary axisx = np.arange(len(names))# Bar width for grouped barsbar_width = 0.4# Plot latency bars in amberbars1 = ax1.bar(x - bar_width/2, latencies, bar_width, color='#fef3c7', edgecolor='#000', label='Cold Start (s)')# Set y-axis label for latencyax1.set_ylabel('Cold Start Latency (seconds)', color='#991b1b')# Color the tick labels to matchax1.tick_params(axis='y', labelcolor='#991b1b')# Create secondary axis for costax2 = ax1.twinx()# Plot cost bars in blue on secondary axisbars2 = ax2.bar(x + bar_width/2, costs, bar_width, color='#dbeafe', edgecolor='#000', label='Monthly Cost ($)')# Set y-axis label for costax2.set_ylabel('Monthly Cost ($)', color='#2563eb')# Color the tick labels to matchax2.tick_params(axis='y', labelcolor='#2563eb')# Set x-axis tick labels to strategy namesax1.set_xticks(x)ax1.set_xticklabels(names, rotation=15, ha='right')ax1.set_title(f'Cold Start Mitigation: Latency vs Cost (Llama 8B FP16, {TARGET_MODEL_GB}GB)')# Add value annotations on barsfor i, (lat, cost) in enumerate(zip(latencies, costs)):    # Annotate latency bar with value    ax1.text(i - bar_width/2, lat + 0.5, f'{lat:.0f}s', ha='center', fontsize=9)    # Annotate cost bar with value    ax2.text(i + bar_width/2, cost + 20, f'${cost:.0f}', ha='center', fontsize=9)# Combine legends from both axeslines1, labels1 = ax1.get_legend_handles_labels()lines2, labels2 = ax2.get_legend_handles_labels()ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')plt.tight_layout()# Save figure for referenceplt.savefig('mitigation_comparison.png', dpi=150, bbox_inches='tight')plt.show()# Print summary recommendationprint(f"\nRecommendation for {TARGET_MODEL_GB}GB model:")print(f"  NVMe cache: best cost/latency tradeoff ({latencies[1]:.0f}s, ${costs[1]}/mo)")print(f"  Pre-warming: zero latency but ${costs[3]:.0f}/mo overhead")

## Experiment 3: Scale-to-Zero Break-Even AnalysisAt what utilization rate does serverless (with cold starts) become more expensive than always-on due to wasted user time?

In [ ]:
# --- Break-even analysis parameters ---# Hourly GPU cost for always-on servingHOURLY_GPU_COST = 3.50# Value of user time lost during cold start ($/second)USER_TIME_VALUE_PER_S = 0.01# Cold start latency (seconds) for the modelCOLD_START_LATENCY_S = 25.0# Requests per cold start (batch of users hitting cold instance)REQUESTS_PER_COLD_START = 1# Range of utilization rates to analyzeUTILIZATION_RANGE = np.linspace(0.01, 1.0, 100)def break_even_analysis(utilization, hourly_cost, cold_latency, user_value, req_per_cold):    """Compute cost difference between always-on vs serverless.        Returns: (always_on_cost, serverless_cost) per hour    """    # Always-on: pay full hourly rate regardless of utilization    always_on = np.full_like(utilization, hourly_cost)    # Serverless: pay only for active time + cold start penalty    # Active compute cost (proportional to utilization)    active_cost = utilization * hourly_cost    # Cold start events per hour (scale-up frequency)    # Assume one cold start per idle->active transition    # Frequency of transitions increases with lower utilization    cold_starts_per_hour = (1 - utilization) * 3  # ~3 transitions/hr at low util    # Total user-time cost from cold starts    cold_start_penalty = cold_starts_per_hour * cold_latency * user_value * req_per_cold    # Total serverless cost    serverless = active_cost + cold_start_penalty    return always_on, serverless# Compute costs across utilization rangealways_on, serverless = break_even_analysis(    UTILIZATION_RANGE, HOURLY_GPU_COST, COLD_START_LATENCY_S,    USER_TIME_VALUE_PER_S, REQUESTS_PER_COLD_START)# Find the break-even utilization point# Where serverless cost exceeds always-on costcrossover_idx = np.argmin(np.abs(always_on - serverless))# Break-even utilization percentagebreak_even_util = UTILIZATION_RANGE[crossover_idx] * 100# Plot the break-even analysisfig_6, ax_6 = plt.subplots(figsize=(10, 5))# Plot always-on cost (flat line)ax_6.plot(UTILIZATION_RANGE * 100, always_on, color='#991b1b', linewidth=2, label='Always-On')# Plot serverless cost (decreasing with utilization)ax_6.plot(UTILIZATION_RANGE * 100, serverless, color='#166534', linewidth=2, label='Serverless')# Mark the break-even pointax_6.axvline(x=break_even_util, color='orange', linestyle='--', alpha=0.7, label=f'Break-even: {break_even_util:.0f}%')# Shade the region where serverless wins (below break-even)ax_6.fill_between(UTILIZATION_RANGE * 100, serverless, always_on,                where=serverless < always_on, alpha=0.1, color='green')# Shade the region where always-on wins (above break-even)ax_6.fill_between(UTILIZATION_RANGE * 100, serverless, always_on,                where=serverless >= always_on, alpha=0.1, color='red')# Axis labels and titleax_6.set_xlabel('GPU Utilization (%)')ax_6.set_ylabel('Effective Hourly Cost ($)')ax_6.set_title('Scale-to-Zero Break-Even: Serverless vs Always-On')ax_6.legend(loc='upper right')ax_6.grid(True, alpha=0.3)plt.tight_layout()# Save figureplt.savefig('break_even_analysis.png', dpi=150, bbox_inches='tight')plt.show()# Print the key findingprint(f"Break-even utilization: {break_even_util:.0f}%")print(f"Below {break_even_util:.0f}% utilization: serverless saves money")print(f"Above {break_even_util:.0f}% utilization: always-on is cheaper")